# Positional Encodings

To give the model a sense of "time" or "order," we have to inject Positional Encodings. 

Without them, the self-attention mechanism is just a "Bag of Words" model—it knows which words are in the sentence, but it doesn't know if "Cat" came before "Sat."

## The Intuition: The Time Stamp

* Since Transformers don't process tokens one-by-one (like RNNs), we have to add a unique signal to each token's embedding that indicates its position.

* Modern LLMs typically use Learned Positional Embeddings (like GPT-3) or Rotary Positional Embeddings (RoPE) (like Llama 3). For learning the fundamentals, the original Sinusoidal Encoding is the most illustrative because it uses fixed math to create a "topographic map" of the sequence.

## The Math

We generate a matrix of the same shape as our word embeddings using sine and cosine functions of different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

We simply add this matrix to our word embeddings: 

$$X_{final} = X_{embeddings} + PE$$


**The variables in those equations:**
* `pos`: The index of the word in the sequence (0, 1, 2, 3...).
* `i`: The index of the dimension within the embedding vector (from 0 up to dmodel​).
* `dmodel`​: The total size of your embedding (e.g., 512).
* `10000`: This is a "base" constant. It’s chosen to be very large so that the wavelengths of the sines and cosines vary significantly across the vector.

As `i` (the dimension index) increases, the denominator gets much larger. In trigonometry, a larger denominator in the argument of a sine function means a longer wavelength (it moves slower).
* At i=0 (beginning of the vector): The wave wiggles very fast. It changes drastically from Word 1 to Word 2.
* At i=dmodel​ (end of the vector): The wave is massive and slow. It might not even complete a full cycle over a sequence of 1,000 words.


| Component | Function | Result |
| :--- | :--- | :--- |
| **$pos$** | Input position | Ensures word 1 $\neq$ word 2. |
| **$2i / d_{model}$** | Frequency control | Creates fast waves and slow waves. |
| **Sine/Cosine** | Mathematical oscillation | Allows for relative distance calculations. |
| **Addition ($+$)** | Combining signals | Keeps meaning and position in one vector. |

## Why does this work?

**Fixed yet Informative:** 
* Because the Sine/Cosine functions have different wavelengths, the model can learn to attend by relative distance (e.g., "Look at the word 3 positions back").

If you give a Transformer the sentences "The dog bit the man" and "The man bit the dog," the raw self-attention mechanism sees them as exactly the same set of vectors. Without a "timestamp," the model treats a sentence like a pile of words in a bag rather than an ordered sequence.

In language, position is meaning. In an RNN, position is implicit because the model sees words one by one. In a Transformer, every word is processed at the same time. We need to "bake" the position into the word's identity before it enters the attention mechanism.

We can't just use a simple integer (1, 2, 3...) because:

* Scale: In long sequences, the integers would get huge, potentially overshadowing the actual word embedding values.

* Generalization: If the model only ever saw sequences of length 500 during training, it wouldn't know how to handle position 501.

### Why Sinusoidal Encodings work

The original Transformer used a clever "clock-like" system of sines and cosines.

#### The Intuition: The Odometer

Imagine an odometer on a car. The "ones" column spins very fast, the "tens" column spins slower, and the "thousands" column barely moves. By looking at the orientation of all the wheels at once, you can determine exactly how far the car has traveled.

Sinusoidal encodings do the same thing:

* **Different Frequencies:** Each dimension of the embedding vector corresponds to a different frequency. Some dimensions change rapidly from word to word (high frequency), while others change very slowly (low frequency).
* **Unique Signatures:** Every position  gets a unique combination of sine/cosine values across the embedding dimensions.

#### The Mathematical "Superpower": Relative Positions

The coolest part about using sines and cosines is that for any fixed offset ,  can be represented as a linear function of .
Mathematically, this means the model can easily learn to attend to **relative** positions (e.g., *"Look at the word exactly 2 places to my left"*), regardless of where in the sentence the words are.

---

### Modern Evolution: Learned & Rotary (RoPE)

While the sine/cosine method is the foundation, modern LLMs have moved toward two other methods:

| Method | How it works | Used in... |
| --- | --- | --- |
| **Learned Embeddings** | Treats "Position 1," "Position 2," etc., as parameters to be learned during training, just like word meanings. | GPT-3, BERT |
| **Rotary (RoPE)** | Instead of adding a vector, it **rotates** the  and  vectors in space. The angle of rotation depends on the position. | Llama 2/3, PaLM, Mistral |

**Why RoPE is winning:** Rotary embeddings (RoPE) are incredibly powerful because they naturally decay as tokens get further apart. This makes it much easier for models to handle "Infinite" context windows or extrapolate to longer sequences than they were trained on.

---

### Implementation Deep Dive (The "Add" Step)

In PyTorch, we don't concatenate the position; we **add** it.

```python
# Embedding: (Batch, Seq_Len, Dim)
# PosEncoding: (1, Seq_Len, Dim)

x = self.embedding(tokens) 
x = x + self.pos_encoding[:, :x.size(1), :] 

```

When you add them, the word's meaning and its position are "intermingled." Because the attention mechanism is a series of linear transformations (the  matrices), the model can learn to "recover" the positional signal from the sum to use it for scoring, while still retaining the semantic meaning of the word.

### Summary

* **Self-attention is blind to order.**
* **Positional Encoding** gives each word a unique "spatial coordinate."
* **Sines/Cosines** allow the model to understand both absolute and relative distance.
* **Modern models (like Llama)** use rotation (RoPE) to make this even more mathematically robust.

Would you like to see how **Rotary Positional Embeddings (RoPE)** look in code, or shall we move on to the **Feed-Forward Networks** that sit on top of this attention layer?

## Pytorch Implementation (Positional Encoding)

In [2]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super().__init__()
        # Create a matrix of shape (max_len, embed_size)
        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # Division term for the frequencies
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size))
        
        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add a batch dimension: (1, max_len, embed_size)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch, seq_len, embed_size)
        # Add the positional encoding to the input embeddings
        return x + self.pe[:, :x.size(1), :]

/Users/jaero/projects/interviews/scale_ai/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
# 1. Setup Parameters
batch_size = 1
seq_len = 3      # "I love AI"
embed_size = 4   # Small embedding for visualization

# 2. Mock Input: Tokenized words (e.g., [5, 22, 9])
# In a real model, these would come from an nn.Embedding layer
input_embeddings = torch.tensor([[
    [0.1, 0.1, 0.1, 0.1], # Word 1: "I"
    [0.2, 0.2, 0.2, 0.2], # Word 2: "love"
    [0.3, 0.3, 0.3, 0.3]  # Word 3: "AI"
]])

# 3. Initialize Positional Encoding
pos_encoder = PositionalEncoding(embed_size=embed_size, max_len=10)

# 4. Apply Encoding
output = pos_encoder(input_embeddings)

print("Original Embeddings Shape:", input_embeddings.shape)
print("Encoded Output Shape:", output.shape)
print("\nFirst word ('I') before PE:\n", input_embeddings[0, 0])
print("First word ('I') after PE:\n", output[0, 0])

Original Embeddings Shape: torch.Size([1, 3, 4])
Encoded Output Shape: torch.Size([1, 3, 4])

First word ('I') before PE:
 tensor([0.1000, 0.1000, 0.1000, 0.1000])
First word ('I') after PE:
 tensor([0.1000, 1.1000, 0.1000, 1.1000])


### The Semantic Meaning (Before)

    First word ('I') before PE: tensor([0.1000, 0.1000, 0.1000, 0.1000])

These numbers represent the raw meaning of the word "I." In a real model, these values would be learned weights that capture "first-person pronoun." Notice that every dimension is identical here (0.1) because we used a mock tensor, making it very easy to see exactly what the math changes.

### The Positional "Stamp" (After)

    First word ('I') after PE: tensor([0.1000, 1.1000, 0.1000, 1.1000])

This is where the magic happens. Let's look at why the numbers became 0.1 and 1.1. This word is at Position 0. If we look at our Sine/Cosine formulas for pos=0:
* Index 0 & 2 (Even): sin(0/divisor)=0.
    - Calculation: 0.1(word)+0(pos)=0.1000
* Index 1 & 3 (Odd): cos(0/divisor)=1.
    - Calculation: 0.1(word)+1(pos)=1.1000


### What does this tell the model

When this tensor reaches the Self-Attention layer:
* The model sees the value 0.1 and recognizes the word "I."
* The model sees the oscillation between 0.1 and 1.1 and says, "Aha! This word is at the very start of the sentence."

## Q: ok but why does it have to use sine/cosine and not just index ?

This is the "million-dollar question" of the original Transformer paper. Using a raw index (0, 1, 2, 3...) seems intuitive, but it fails for three very specific mathematical reasons that would make the model's "brain" break.

### 1. The Scaling Problem (Exploding Values)

Neural networks are very sensitive to the **scale** of numbers. They generally like values between -1 and 1.

* If you use a raw index, and your sentence is 2,000 words long, the first word gets a `0` and the last word gets a `2000`.
* The model would have to deal with massive numbers at the end of the sequence that completely drown out the small embedding values (the `0.1` meanings).
* **Sine/Cosine** are "bounded." No matter how long the book is, the positional encoding will **always be between -1 and 1**.

### 2. The Generalization Problem (The "Infinite" Ruler)

If you train a model on sentences of length 128 using raw indices, it learns that "128" means "The End."

* If you then give it a sentence of length 256 during testing, the model has **no idea** what "129" through "256" mean. It hasn't seen numbers that big.
* **Sine/Cosine** are periodic (they repeat). Because the model learns the *relationship* between the waves (the "shape" of the curve) rather than the absolute number, it can often handle sequences longer than what it saw in training. It understands the "rhythm" of the positions.

### 3. The "Relative" Secret (The Most Important Part)

In language, **relative distance** matters more than **absolute position**.
Knowing that *"Apple"* is the 50th word is less important than knowing that the word *"Red"* is **exactly 1 spot before it**.

Thanks to a specific trigonometric identity:


Because of this, the position at  (a word  steps away) can be calculated as a **linear transformation** of the position at .

**In plain English:** The model’s weight matrices can learn a single "rule" (a linear shift) that works the same way whether it's looking 2 words ahead at the beginning of a sentence or 2 words ahead at the end of a sentence. With a raw index, the difference between  and  is mathematically very different from the difference between  and . With sine waves, the "angle" of change is identical.

---

### Summary: Index vs. Sine/Cosine

| Feature | Raw Index (1, 2, 3...) | Sine/Cosine Encoding |
| --- | --- | --- |
| **Range** | Unbounded (can be 0 to ) | Always -1 to 1 |
| **Scale** | Can overwhelm word meaning | Stays in the same range as words |
| **Relative Distance** | Hard for the model to "math" out | Simple linear rotation |
| **Long sequences** | Fails on unseen lengths | Generalizes much better |